In [1]:
# --- Path setup ---
import sys
import os
import math
import time
import contextlib
import io
import warnings
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "max_k_cut").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

# --- Scientific / data ---
import networkx as nx
import numpy as np
import scipy.io

# --- Visualization ---
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams["figure.dpi"] = 1000
mpl.rcParams["savefig.dpi"] = 1000

# --- Qiskit ecosystem ---
import qiskit
import qiskit_aer
import qiskit_algorithms
import qiskit_optimization
from qiskit import QuantumCircuit
from qiskit.primitives import Sampler
from qiskit.circuit.library import QAOAAnsatz, RYGate, XGate, CXGate
from qiskit.visualization import plot_histogram, plot_state_city, plot_state_qsphere, plot_bloch_multivector, plot_distribution
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

from qiskit_algorithms import QAOA, SamplingVQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA

from qiskit_optimization.algorithms import MinimumEigenOptimizer, SolutionSample, OptimizationResultStatus
from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import LinearEqualityToPenalty, LinearInequalityToPenalty, QuadraticProgramToQubo
from qiskit_optimization.translators import from_docplex_mp

# --- IPython ---
from IPython.display import display, Math

# --- Project ---
from max_k_cut import (
    docplex_BQO, docplex_RBQO, docplex_QUBO, docplex_RQUBO, docplex_QUBO_no_constraints,
    tight_qubo_penalty, tight_rqubo_penalty, naive_qubo_penalty, naive_rqubo_penalty,
    interpolated_qubo_penalty, interpolated_rqubo_penalty,
    generate_graph, plot_graph, feasibility_filter, expected_value,
    create_dicke_initial_state, create_full_xy_mixer, create_ring_xy_mixer,
)
from max_k_cut.qaoa import run_qaoa_extract_samples
from scripts.plot_histogram import plot_qaoa_histogram

warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- Version info ---
print("qiskit version:", qiskit.__version__)
print("qiskit aer version:", qiskit_aer.__version__)
print("qiskit algorithms version:", qiskit_algorithms.__version__)
print("qiskit optimization version:", qiskit_optimization.__version__)

# %config InlineBackend.figure_format = 'retina'

qiskit version: 1.3.2
qiskit aer version: 0.15.1
qiskit algorithms version: 0.3.1
qiskit optimization version: 0.6.1


In [2]:
def add_a_weighted_edge(graph, weight):

    # Get all edge data
    edges = list(graph.edges(data=True))

    # Extract edge weights and check if graph is weighted
    weights = [d['weight'] for (_, _, d) in edges]

    # pick a random edge to add weight to
    random_edge = np.random.choice(len(weights))

    # Check if graph is weighted
    (u, v, _) = edges[random_edge]
    graph.edges[u, v]['weight'] = weight

    # Add weight
    # weights[random_edge] = weight

    return graph


test_graph = generate_graph(6, 0.5, False)
test_graph_weighted = add_a_weighted_edge(test_graph, 10)
# plot_graph(test_graph)
test_graph.edges(data=True)

EdgeDataView([(0, 1, {'weight': 1}), (0, 3, {'weight': 1}), (0, 4, {'weight': 1}), (1, 2, {'weight': 1}), (3, 4, {'weight': 10}), (4, 5, {'weight': 1})])

# Create QAOA models for enforcing constraints using penalties, mixers, and a combination of the two

In [3]:
# Experiment parameters
K = 3 # hamming weight constraint
num_nodes = 6 # number of nodes in the graph
edge_probability = 0.5
weighted = False
weight_range = 1


# Get the noise model and coupling map from the backend
backend = FakeBrisbane()
backend = AerSimulator.from_backend(backend)
# backend = AerSimulator()
sampler = Sampler()
shots = 10000
sampler.set_options(shots=shots, backend=backend)
optimizer = COBYLA()
reps = 4
initial_point = np.random.rand(2 * reps) * np.pi / 2 # initial point for the optimizer

In [4]:
# ============================Penalty QAOA============================
penalty_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps, 
    initial_point=initial_point,
    )

# create minimum eigen optimizer based on solver used
penalty_qaoa_optimizer = MinimumEigenOptimizer(penalty_qaoa)

In [5]:
# ============================Dicke-State Penalty QAOA============================
# use with qubo model for equality constraints
init_qc = create_dicke_initial_state(num_nodes, K)

dicke_penalty_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=init_qc,
    initial_point=initial_point,
    )

# create minimum eigen optimizer based on solver used
dicke_penalty_qaoa_optimizer = MinimumEigenOptimizer(dicke_penalty_qaoa)
#init_qc.draw(output='mpl')


In [6]:
# ============================Dicke-State Mixer QAOA============================
from qiskit.circuit import Parameter

init_qc = create_dicke_initial_state(num_nodes, K)
beta = Parameter("β")
mixer = create_ring_xy_mixer(num_nodes, K, beta)
# mixer = create_full_xy_mixer(num_nodes, K, beta)

mixer_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=init_qc,
    mixer=mixer,
    initial_point=initial_point,
    )

# create minimum eigen optimizer based on solver used
mixer_qaoa_optimizer = MinimumEigenOptimizer(mixer_qaoa)

mixer.draw(output='mpl', fold=-1)

In [7]:
# ============================Classical Solver============================
np_solver = NumPyMinimumEigensolver() # exact classical solver
np_optimizer = MinimumEigenOptimizer(np_solver)

In [8]:
# penalty_list = np.linspace(0, 1, 6)
# num_graphs = 20

penalty_list = np.linspace(0, 1, 6)
num_graphs = 20

QUBO_EXP_VALS     = np.zeros((num_graphs, len(penalty_list)))
QUBO_FEASIBILITY  = np.zeros((num_graphs, len(penalty_list)))
RQUBO_EXP_VALS    = np.zeros((num_graphs, len(penalty_list)))
RQUBO_FEASIBILITY = np.zeros((num_graphs, len(penalty_list)))

DICKE_QUBO_EXP_VALS     = np.zeros((num_graphs, len(penalty_list)))
DICKE_QUBO_FEASIBILITY  = np.zeros((num_graphs, len(penalty_list)))
PENALTY_MIXER_QUBO_EXP_VALS     = np.zeros((num_graphs, len(penalty_list)))
PENALTY_MIXER_QUBO_FEASIBILITY  = np.zeros((num_graphs, len(penalty_list)))

XY_MIXER_EXP_VALS = np.zeros(num_graphs)
XY_MIXER_FEASIBILITY = np.zeros(num_graphs)

for graph in range(num_graphs):
    # generate random graph
    G = generate_graph(num_nodes, edge_probability, weighted, weight_range)
    # add a random edge with weight 100
    G = add_a_weighted_edge(G, 10)

    with contextlib.redirect_stdout(io.StringIO()):
        dp_qubo_no_constraints = docplex_QUBO_no_constraints(G, K, "Max-K-Cut")
        mixer_results = run_qaoa_extract_samples(
            [dp_qubo_no_constraints],
            ["XY Mixer"],
            optimizer=mixer_qaoa_optimizer
        )

    xy_samples = mixer_results["XY Mixer"]["samples"]
    xy_filtered, xy_feas_prob = feasibility_filter(G, K, xy_samples, "QUBO (XY Mixer)")
    xy_exp_val = expected_value(xy_filtered)
    XY_MIXER_EXP_VALS[graph] = xy_exp_val
    XY_MIXER_FEASIBILITY[graph] = xy_feas_prob

    print(f"Graph {graph+1}/{num_graphs} XY Mixer:")
    print(f"  Expected Value: {xy_exp_val:.4f}, Feasibility Probability: {xy_feas_prob:.4f}")

    for pen_index, pen in enumerate(penalty_list):
        # Create docplex models
        dp_qubo_interp = docplex_QUBO(G, K, interpolated_qubo_penalty(G, K, pen), "Max-K-Cut")
        dp_rqubo_interp = docplex_RQUBO(G, K, interpolated_rqubo_penalty(G, K, pen), "Max-K-Cut")

        with contextlib.redirect_stdout(io.StringIO()):
            # Run QAOA
            penalty_results = run_qaoa_extract_samples(
                [dp_qubo_interp, dp_rqubo_interp],
                ["QUBO (Interpolated)", "RQUBO (Interpolated)"],
                optimizer=penalty_qaoa_optimizer
            )
            dicke_results = run_qaoa_extract_samples(
                [dp_qubo_interp],
                ["QUBO (Interpolated)"],
                optimizer=dicke_penalty_qaoa_optimizer
            )
            penalty_mixer_results = run_qaoa_extract_samples(
                [dp_qubo_interp],
                ["QUBO (Interpolated)"],
                optimizer=mixer_qaoa_optimizer
            )

        # Retrieve the raw samples
        samples_qubo = penalty_results["QUBO (Interpolated)"]["samples"]
        samples_rqubo = penalty_results["RQUBO (Interpolated)"]["samples"]
        samples_dicke_qubo = dicke_results["QUBO (Interpolated)"]["samples"]
        samples_penalty_mixer_qubo = penalty_mixer_results["QUBO (Interpolated)"]["samples"]

        # Filter the samples and get the feasibility probability.
        filtered_samples_qubo, feas_prob_qubo = feasibility_filter(G, K, samples_qubo, "QUBO (Interpolated)")
        filtered_samples_rqubo, feas_prob_rqubo = feasibility_filter(G, K, samples_rqubo, "RQUBO (Interpolated)")
        filtered_samples_dicke, feas_prob_dicke = feasibility_filter(G, K, samples_dicke_qubo, "QUBO (Interpolated)")
        filtered_samples_penalty_mixer, feas_prob_penalty_mixer = feasibility_filter(G, K, samples_penalty_mixer_qubo, "QUBO (Interpolated)")

        # Calculate the expected value of the objective function.
        exp_val_qubo = expected_value(filtered_samples_qubo)
        exp_val_rqubo = expected_value(filtered_samples_rqubo)
        exp_val_dicke = expected_value(filtered_samples_dicke)
        exp_val_penalty_mixer = expected_value(filtered_samples_penalty_mixer)

        # Store the results
        QUBO_EXP_VALS[graph, pen_index] = exp_val_qubo
        QUBO_FEASIBILITY[graph, pen_index] = feas_prob_qubo

        RQUBO_EXP_VALS[graph, pen_index] = exp_val_rqubo
        RQUBO_FEASIBILITY[graph, pen_index] = feas_prob_rqubo

        DICKE_QUBO_EXP_VALS[graph, pen_index] = exp_val_dicke
        DICKE_QUBO_FEASIBILITY[graph, pen_index] = feas_prob_dicke

        PENALTY_MIXER_QUBO_EXP_VALS[graph, pen_index] = exp_val_penalty_mixer
        PENALTY_MIXER_QUBO_FEASIBILITY[graph, pen_index] = feas_prob_penalty_mixer

        # Print the results
        print(f"Graph {graph+1}/{num_graphs}, Penalty {pen:.2f}:")
        print(f"  QUBO Expected Value: {exp_val_qubo:.4f}, Feasibility Probability: {feas_prob_qubo:.4f}")
        print(f"  RQUBO Expected Value: {exp_val_rqubo:.4f}, Feasibility Probability: {feas_prob_rqubo:.4f}")
        print(f"  Dicke QUBO Expected Value: {exp_val_dicke:.4f}, Feasibility Probability: {feas_prob_dicke:.4f}")
        print(f"  Penalty+Mixer QUBO Expected Value: {exp_val_penalty_mixer:.4f}, Feasibility Probability: {feas_prob_penalty_mixer:.4f}")

# save results
os.makedirs("Data", exist_ok=True)
np.savez(
    os.path.join("Data", "Noisy_Results_P4_random_init_20_graphs.npz"),
    QUBO_EXP_VALS=QUBO_EXP_VALS,
    QUBO_FEASIBILITY=QUBO_FEASIBILITY,
    RQUBO_EXP_VALS=RQUBO_EXP_VALS,
    RQUBO_FEASIBILITY=RQUBO_FEASIBILITY,
    DICKE_QUBO_EXP_VALS=DICKE_QUBO_EXP_VALS,
    DICKE_QUBO_FEASIBILITY=DICKE_QUBO_FEASIBILITY,
    PENALTY_MIXER_QUBO_EXP_VALS=PENALTY_MIXER_QUBO_EXP_VALS,
    PENALTY_MIXER_QUBO_FEASIBILITY=PENALTY_MIXER_QUBO_FEASIBILITY,
    XY_MIXER_EXP_VALS=XY_MIXER_EXP_VALS,
    XY_MIXER_FEASIBILITY=XY_MIXER_FEASIBILITY
)

Probability of feasible solution for QUBO (XY Mixer): 1.0000
Graph 1/20 XY Mixer:
  Expected Value: 0.7711, Feasibility Probability: 1.0000
Probability of feasible solution for QUBO (Interpolated): 0.0054
Probability of feasible solution for RQUBO (Interpolated): 0.2053
Probability of feasible solution for QUBO (Interpolated): 0.0040
Probability of feasible solution for QUBO (Interpolated): 1.0000
Graph 1/20, Penalty 0.00:
  QUBO Expected Value: 0.6045, Feasibility Probability: 0.0054
  RQUBO Expected Value: 0.7956, Feasibility Probability: 0.2053
  Dicke QUBO Expected Value: 0.7625, Feasibility Probability: 0.0040
  Penalty+Mixer QUBO Expected Value: 0.7737, Feasibility Probability: 1.0000
Probability of feasible solution for QUBO (Interpolated): 0.0078
Probability of feasible solution for RQUBO (Interpolated): 0.2687
Probability of feasible solution for QUBO (Interpolated): 0.0093
Probability of feasible solution for QUBO (Interpolated): 1.0000
Graph 1/20, Penalty 0.20:
  QUBO Expect